In [227]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [228]:
train_df = pd.read_csv('train_c.csv')
test_df = pd.read_csv('test_c.csv')
X = train_df.drop(["임신 성공 여부"], axis=1) 
y = train_df["임신 성공 여부"]

In [229]:
categorical_cols = X.select_dtypes(include=['object']).columns
categorical_cols

Index(['시술 시기 코드', '시술 당시 나이', '시술 유형', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유',
       '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수', '총 임신 횟수',
       'IVF 임신 횟수', 'DI 임신 횟수', '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수', '난자 출처',
       '정자 출처', '난자 기증자 나이', '정자 기증자 나이', '배아 이식 경과일'],
      dtype='object')

In [230]:
numerical_cols = X.select_dtypes(include=['number']).columns
numerical_cols

Index(['배란 자극 여부', '단일 배아 이식 여부', '착상 전 유전 진단 사용 여부', '남성 주 불임 원인',
       '남성 부 불임 원인', '여성 주 불임 원인', '여성 부 불임 원인', '부부 주 불임 원인', '부부 부 불임 원인',
       '불명확 불임 원인', '불임 원인 - 난관 질환', '불임 원인 - 남성 요인', '불임 원인 - 배란 장애',
       '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제', '불임 원인 - 자궁내막증', '불임 원인 - 정자 농도',
       '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태', '총 생성 배아 수',
       '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수',
       '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수',
       '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수', '동결 배아 사용 여부',
       '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부', '난자 혼합 경과일'],
      dtype='object')

In [231]:
train_df = train_df.drop(columns=['불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인',
       'IVF 시술 횟수', 'DI 시술 횟수','IVF 임신 횟수', 'DI 임신 횟수','IVF 출산 횟수', 'DI 출산 횟수',
       '배아 생성 주요 이유', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수'
       ])
test_df = test_df.drop(columns=['불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인',
       'IVF 시술 횟수', 'DI 시술 횟수','IVF 임신 횟수', 'DI 임신 횟수','IVF 출산 횟수', 'DI 출산 횟수',
       '배아 생성 주요 이유', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수'
       ])

In [232]:
ordinal_cols = {
    "시술 당시 나이":['알 수 없음','만18-34세','만35-37세','만38-39세','만40-42세','만43-44세','만45-50세'], 
    "총 시술 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    "클리닉 내 총 시술 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    #"IVF 시술 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    #"DI 시술 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    "총 임신 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    #"IVF 임신 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    #"DI 임신 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'], 
    "총 출산 횟수":['0회','1회','2회','3회','4회','5회','6회 이상'],
    #"IVF 출산 횟수":['0회','1회','2회','3회','4회','5회'], 
    #"DI 출산 횟수":['0회','1회','2회','3회','4회','5회'], 
    "난자 기증자 나이":['알 수 없음','만20세 이하','만21-25세','만26-30세','만31-35세'], 
    "정자 기증자 나이":['알 수 없음','만20세 이하','만21-25세','만26-30세','만31-35세','만36-40세','만41-45세'],
    "배아 이식 경과일":['0일','1-3일','4-6일','7일']
}

onehot_cols = [
    "시술 시기 코드", "시술 유형", "특정 시술 유형", "배란 유도 유형",
    #"배아 생성 주요 이유", 
    "난자 출처", "정자 출처"
]

In [233]:
ordinal_encoder_dict = {}
for col, categories in ordinal_cols.items():
    encoder = OrdinalEncoder(categories=[categories], handle_unknown="use_encoded_value", unknown_value=-1)
    
    train_df[col] = encoder.fit_transform(train_df[[col]])
    test_df[col] = encoder.transform(test_df[[col]])
    
    ordinal_encoder_dict[col] = encoder  # 인코더 저장

# One-Hot Encoding 적용 (순서가 없는 범주형 컬럼)
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

train_encoded = onehot_encoder.fit_transform(train_df[onehot_cols])
test_encoded = onehot_encoder.transform(test_df[onehot_cols])

encoded_columns = onehot_encoder.get_feature_names_out(onehot_cols)

# DataFrame 변환
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_columns, index=train_df.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_columns, index=test_df.index)

# 기존 데이터에서 원핫 인코딩한 컬럼 제거 후 추가
train_df = train_df.drop(columns=onehot_cols).reset_index(drop=True).join(train_encoded_df)
test_df = test_df.drop(columns=onehot_cols).reset_index(drop=True).join(test_encoded_df)


In [234]:
train_df.to_csv("train_e.csv", index=False, encoding='utf-8-sig')
test_df.to_csv("test_e.csv", index=False, encoding='utf-8-sig')